# v27 — the v19 we PROMISED but never built (fundamentals)

**v23 LB just landed at 0.7154** — same recipe as v19 (LB 0.7455), only difference is `BASE_SEED=2`.
That's a 0.03 spread from a SEED CHANGE on the same recipe, and we have **no validation tracking
to find the actually-good checkpoints**. We've been training blind.

## What v19's title claimed but the code didn't do

| Promised in v19's title | v19 reality | v27 fix |
|---|---|---|
| Validation set + best-val ckpt (L4 "always do this") | ✗ saves last epoch | ✓ holds out 2 patients, tracks val AUC, saves best |
| Discriminative LR (standard practice: "1/10 of original LR") | ✗ flat LR everywhere | ✓ backbone LR = head LR × 0.1 |
| Early stopping (L4) | ✗ trains all 12 epochs blind | ✓ saves best-val ckpt across 15 epochs |

This isn't optional polish — it's the standard "always do this" list, standard practice.

## Why this matters more than ensembling

| Experiment | LB |
|---|---|
| v19 (EffNet-B0, seed=1, no val tracking) | **0.7455** |
| v23 (same recipe, seed=2) | 0.7154 |
| v22 sigmoid_avg (v19 + v21) | 0.7422 |
| v22 rank_avg (v19 + v21) | 0.7436 |
| v22 geomean (v19 + v21) | 0.7328 |
| v21 (ResNet-50 + 224 upscale) | 0.7018 |

Every ensemble we built underperformed v19. The 0.03 seed-spread tells us v19 is a lucky draw,
not a stable result. Fundamentals first.

## Single change vs v19 = the training loop

| Lever | v19 | v27 |
|---|---|---|
| Backbone | EfficientNet-B0 | **same** (configurable below for v28/v29) |
| MIL loss + aug + AdaBN + TTA + stain norm | yes | **same** |
| Train data | all 12 patients | **10 patients** (val = pat_5 + pat_14 held out) |
| Optimizer | AdamW, single LR=3e-4 | **AdamW, 2 groups: backbone=3e-5, head=3e-4** |
| Scheduler | OneCycleLR | **OneCycleLR with per-group max_lr** |
| Epochs | 12 | **15** (extra slack for late convergence) |
| Checkpoint saved | last epoch | **best val_auc_cell across all epochs** |

Held-out val patients: **pat_5 (cancer, hardest)** + **pat_14 (healthy, borderline)** — class-balanced,
historically the most failure-prone patients per past experiments. Cell-level val AUC is the primary
metric (~20k val cells gives a meaningful number; patient-level AUC across 2 patients is just binary).

## Compute on T4

| Stage | Time |
|---|---|
| JPEG cache (one-time) | ~56 min |
| Pixel stats | ~30s |
| Train 15 epochs with val pass | ~75 min |
| 8-way D4 TTA + AdaBN | ~7 min |
| **Total** | **~2h 20min** |

About 15 min more than v19. Within Kaggle's 9h ceiling.

## Backbone is configurable for v28/v29

Set `BACKBONE = "resnet50"` or `"densenet201"` in the config cell to clone v27 → v28 / v29
and test the other recommended backbones under the same proper L4 recipe. That's
the proper way to test those architectures — not v21's confounded upscale, but with val
tracking and disc LR like the teacher said.

## Reading the v27 LB

| LB outcome | Interpretation |
|---|---|
| **≥ 0.77** | fundamentals were the unlock. We're competitive. Clone to v28/v29 to test backbones. |
| **0.75–0.77** | Solid +0.01–0.02 over v19. Definitely the right path. Continue. |
| **0.73–0.75** | Mild improvement. May be that 2-patient val is too noisy. Try different val patients. |
| **< 0.73** | Holding out val data hurt more than disc LR helped. v28 = refit on full data with discovered best epoch. |

## IMPORTANT — DO NOT JUST CLICK RUN ALL

To get a `submission.csv` you must:

1. Click **Save Version** (top-right green button)
2. Choose **Save & Run All (Commit)**
3. Description: `v27: fundamentals (val + disc LR + best-val ckpt) — EffNet-B0`
4. **Wait for the commit to finish** (~2h 20min — you'll get a notification)
5. Open the saved version → Output tab → submission.csv is there

Running cells one-by-one in editor mode trains the model but does NOT persist outputs as a Saved Version.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === v27 BACKBONE selection — change this single line for v28/v29 ===
# One of: "efficientnet_b0" (v19/v27 default), "resnet50" (teacher standard practice),
#         "densenet201" (teacher standard practice alt), "resnet18" (floor).
BACKBONE = "efficientnet_b0"

# === v27 NEW: fundamentals ===
# Held-out validation patients (class-balanced: 1 cancer + 1 healthy).
# pat_5  = cancer, historically the v16 failure case
# pat_14 = healthy, historically the v16 borderline case
VAL_PATIENTS         = [5, 14]
BACKBONE_LR_RATIO    = 0.1       # standard practice: "1/10 of original LR is good starting point"
BEST_VAL_METRIC      = "val_auc_cell"  # cell-level val AUC is the meaningful one (~20k val cells)

# === Inherited from v19 (unchanged) ===
USE_MIL_LOSS        = True
MIL_WEIGHT          = 0.5
USE_STRONG_AUG      = True
RANDOM_ERASING_P    = 0.25

USE_TEST_STAIN_NORM = True
USE_ADABN           = True
USE_MULTISCALE_TTA  = False
TTA_SCALES          = (112, 128, 144)

LABEL_SMOOTHING     = 0.0

# DenseNet-only safety (gradient checkpointing in dense blocks).
DENSENET_MEMORY_EFFICIENT = True

# === Training ===
BASE_SEED   = 1     # same seed as v19 — direct comparability of recipe change
EPOCHS      = 15    # v19 used 12; we add 3 epochs of slack since we save best-val ckpt
BATCH_SIZE  = 128
LR          = 3e-4  # head LR (backbone gets LR * BACKBONE_LR_RATIO = 3e-5)
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.0
DROPOUT     = 0.3

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# v11 hardcoded stats (used when USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

_VALID_BACKBONES = {"efficientnet_b0", "resnet50", "densenet201", "resnet18"}
assert BACKBONE in _VALID_BACKBONES, \
    f"Unknown BACKBONE={BACKBONE!r}, must be one of {_VALID_BACKBONES}"

print(f"\nConfig (v27 — fundamentals on top of v19 recipe):")
print(f"  BACKBONE             = {BACKBONE}")
print(f"  VAL_PATIENTS         = {VAL_PATIENTS}   (held out from training)")
print(f"  BACKBONE_LR_RATIO    = {BACKBONE_LR_RATIO}  (head LR={LR}, backbone LR={LR*BACKBONE_LR_RATIO})")
print(f"  BEST_VAL_METRIC      = {BEST_VAL_METRIC!r}  -> save ckpt only when this improves")
print(f"  USE_MIL_LOSS         = {USE_MIL_LOSS}  weight={MIL_WEIGHT}")
print(f"  USE_STRONG_AUG       = {USE_STRONG_AUG}  erasing_p={RANDOM_ERASING_P}")
print(f"  USE_TEST_STAIN_NORM  = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN            = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA   = {USE_MULTISCALE_TTA}  scales={TTA_SCALES}")
print(f"  EPOCHS               = {EPOCHS}  BATCH_SIZE = {BATCH_SIZE}  BASE_SEED = {BASE_SEED}")
print(f"  MIXUP_ALPHA          = {MIXUP_ALPHA}  (disabled)")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
# === Backbone builders — one per supported architecture, dispatched by BACKBONE string ===

def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features  # 2048
    net.fc = nn.Identity()
    return net, fd

def _make_densenet201_branch(pretrained=True):
    """torchvision densenet201 with 1-ch stem + memory_efficient gradient checkpointing."""
    weights = "DEFAULT" if pretrained else None
    net = models.densenet201(weights=weights, memory_efficient=DENSENET_MEMORY_EFFICIENT)
    old = net.features.conv0
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.features.conv0 = new_conv
    fd = net.classifier.in_features  # 1920
    net.classifier = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

_BACKBONE_FACTORY = {
    "resnet18":        _make_resnet18_branch,
    "resnet50":        _make_resnet50_branch,
    "densenet201":     _make_densenet201_branch,
    "efficientnet_b0": _make_effnet_b0_branch,
}

def make_branch(backbone, pretrained=True):
    if backbone not in _BACKBONE_FACTORY:
        raise ValueError(f"Unknown backbone {backbone!r}")
    return _BACKBONE_FACTORY[backbone](pretrained)


class MultimodalClassifier(nn.Module):
    """Dual-branch (BF + FL) classifier with late concat fusion. `backbone` selects arch."""
    def __init__(self, backbone, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.backbone_name = backbone
        self.bf_branch, fd = make_branch(backbone, pretrained)
        self.fl_branch, _  = make_branch(backbone, pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(backbone=BACKBONE, pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Output shape: {_m(_x, _x).shape}   params: {n_params / 1e6:.1f}M")
    print(f"Backbone: {BACKBONE}")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + small rotation, applied identically to BF and FL (paired)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        # v19 NEW: paired affine — same translate applied to both modalities so
        # BF/FL stay registered.
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        return bf, fl

def train_modality_transform(modality):
    """v19: stronger color jitter, with optional RandomErasing applied after normalization."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
        if RANDOM_ERASING_P > 0:
            # RandomErasing operates on normalized tensors; value=0 means it erases to the
            # normalized 0 (which corresponds to original-pixel = mean).
            steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                         ratio=(0.3, 3.3), value=0.0))
        return T.Compose(steps)
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    """v19 paired aug: D4 + ±10° rot, plus ±15° affine and 10% translate when strong aug is on."""
    if USE_STRONG_AUG:
        return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10)
    return PairedGeoAug(max_rot=10.0)

print(f"Augmentation summary:")
print(f"  ColorJitter:      {'brightness/contrast=0.4' if USE_STRONG_AUG else 'brightness/contrast=0.2'}")
print(f"  RandomErasing:    p={RANDOM_ERASING_P if USE_STRONG_AUG else 0.0}")
print(f"  PairedGeoAug:     D4 + ±10° rot" + (" + ±15° affine + 10% translate" if USE_STRONG_AUG else ""))

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """Per-patient mean-logit BCE loss (averages cell-logits within each patient)."""
    unique_pids = torch.unique(patient_ids)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits, p_labels = [], []
    for pid in unique_pids:
        mask = patient_ids == pid
        p_logits.append(logits[mask].mean())
        p_labels.append(y[mask][0])
    p_logits = torch.stack(p_logits); p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, log_every=200):
    model.train()
    losses, hard_ys, ps = [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        y_s = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss_cell = criterion_cell(logits, y_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            old_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale: sched.step()
        else:
            loss.backward()
            if GRAD_CLIP > 0: nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); sched.step()
        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f} "
                  f"mil {float(np.mean(mil_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), float(np.mean(cell_losses)), float(np.mean(mil_losses)), auc


# === v27 NEW: validation pass — returns rich dict for diagnostics ===
@torch.no_grad()
def run_epoch_val(model, loader):
    """Returns dict with cell-level AUC, patient-level AUC, per-patient stats,
    and raw arrays (cell_preds/cell_labels/cell_pids in val_loader order)."""
    model.eval()
    ys, ps, pids = [], [], []
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(bf, fl)
            p = torch.sigmoid(logits).float()
        ys.append(batch["label"].numpy())
        ps.append(p.cpu().numpy())
        pids.append(batch["patient_id"].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); pids = np.concatenate(pids)
    auc_cell = roc_auc_score(ys, ps) if len(np.unique(ys)) > 1 else float("nan")
    per_patient = {}
    for pid in np.unique(pids):
        mask = pids == pid
        per_patient[int(pid)] = {
            "label": int(ys[mask][0]),
            "mean_prob": float(ps[mask].mean()),
            "std_prob":  float(ps[mask].std()),
            "n_cells":   int(mask.sum()),
        }
    pat_pred  = np.array([per_patient[pid]["mean_prob"] for pid in sorted(per_patient)])
    pat_label = np.array([per_patient[pid]["label"]     for pid in sorted(per_patient)])
    auc_pat = (roc_auc_score(pat_label, pat_pred)
               if len(np.unique(pat_label)) > 1 else float("nan"))
    return {
        "auc_cell":    auc_cell,
        "auc_pat":     auc_pat,
        "per_patient": per_patient,
        "cell_preds":  ps,
        "cell_labels": ys,
        "cell_pids":   pids,
    }


# === Patient split: train vs val ===
val_pids_set = set(VAL_PATIENTS)
all_pids = sorted(df_train["patient_id"].unique())
present_val = sorted([p for p in VAL_PATIENTS if p in all_pids])
missing_val = [p for p in VAL_PATIENTS if p not in all_pids]
if missing_val:
    print(f"  WARNING: requested VAL_PATIENTS not in train: {missing_val}")
assert len(present_val) >= 2, \
    f"Need at least 2 val patients in data, got {present_val} (avail: {all_pids})"

df_train_only = df_train[~df_train["patient_id"].isin(val_pids_set)].reset_index(drop=True)
df_val        = df_train[ df_train["patient_id"].isin(val_pids_set)].reset_index(drop=True)
val_label_by_pid = df_val.groupby("patient_id")["Diagnosis"].first().to_dict()
print(f"\nValidation split:")
print(f"  Train patients: {sorted(df_train_only['patient_id'].unique())}  "
      f"cells={len(df_train_only)}  pos_rate={df_train_only['Diagnosis'].mean():.4f}")
print(f"  Val patients:   {sorted(df_val['patient_id'].unique())}  cells={len(df_val)}  "
      f"labels={val_label_by_pid}")


# === Diagnostic 7: sample BF/FL image preview (uses its own RNG; does not affect training) ===
try:
    preview_rng = np.random.default_rng(42)
    train_pats = sorted(df_train_only["patient_id"].unique())
    train_pat_labels = df_train_only.groupby("patient_id")["Diagnosis"].first().to_dict()
    train_cancer  = [p for p in train_pats if train_pat_labels[p] == 1]
    train_healthy = [p for p in train_pats if train_pat_labels[p] == 0]

    rows = []
    # Val patients first, so we can eyeball whether they look like the rest of the data.
    for pid in present_val:
        rows.append((pid, "VAL", val_label_by_pid[pid]))
    if train_cancer:
        rows.append((int(preview_rng.choice(train_cancer)),  "train", 1))
    if train_healthy:
        rows.append((int(preview_rng.choice(train_healthy)), "train", 0))

    n_per_pat = 2
    fig, axes = plt.subplots(len(rows), n_per_pat * 2,
                             figsize=(n_per_pat * 4, len(rows) * 2.2))
    if len(rows) == 1:
        axes = axes[np.newaxis, :]
    for row, (pid, split, lbl) in enumerate(rows):
        sub = df_train[df_train["patient_id"] == pid]
        sel_idxs = preview_rng.choice(sub.index.values,
                                       size=min(n_per_pat, len(sub)),
                                       replace=False)
        lbl_str = "CANCER" if lbl == 1 else "HEALTHY"
        for col, idx in enumerate(sel_idxs):
            name = df_train.loc[idx, "Name"]
            bf_img = np.asarray(Image.open(io.BytesIO(bf_train_cache[name])).convert("L"))
            fl_img = np.asarray(Image.open(io.BytesIO(fl_train_cache[name])).convert("L"))
            axes[row, col*2  ].imshow(bf_img, cmap="gray", vmin=0, vmax=255)
            axes[row, col*2  ].axis("off")
            axes[row, col*2  ].set_title(f"pat_{pid} ({split} {lbl_str})\nBF",
                                          fontsize=8)
            axes[row, col*2+1].imshow(fl_img, cmap="gray", vmin=0, vmax=255)
            axes[row, col*2+1].axis("off")
            axes[row, col*2+1].set_title("FL", fontsize=8)
    plt.suptitle("Sample BF/FL cells — first rows are held-out VAL patients",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_images.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  Saved sample_images.png  ({len(present_val)} val + {len(rows)-len(present_val)} train rows)")
except Exception as _e:
    print(f"  [warn] image preview skipped: {_e}")


# === Build datasets and loaders ===
print(f"\n=== v27: Training (backbone={BACKBONE}, val={present_val}, {EPOCHS} epochs) ===")
seed_everything(BASE_SEED + 100)
train_ds = CachedCellDataset(df_train_only, bf_train_cache, fl_train_cache,
                             train_modality_transform("bf"),
                             train_modality_transform("fl"),
                             paired_tf=build_paired_aug())
sampler = PatientBalancedSampler(df_train_only, batch_size=BATCH_SIZE,
                                 patients_per_batch=PATIENTS_PER_BATCH, seed=BASE_SEED + 100)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

val_ds = CachedCellDataset(df_val, bf_train_cache, fl_train_cache,
                           eval_modality_transform("bf"),
                           eval_modality_transform("fl"))
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# === Build model with discriminative LR (standard practice: backbone LR = head LR / 10) ===
model = MultimodalClassifier(backbone=BACKBONE, pretrained=True, dropout=DROPOUT).to(DEVICE)
pos = (df_train_only["Diagnosis"] == 1).sum()
neg = (df_train_only["Diagnosis"] == 0).sum()
pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)

backbone_params = list(model.bf_branch.parameters()) + list(model.fl_branch.parameters())
head_params = list(model.head.parameters())
n_back = sum(p.numel() for p in backbone_params)
n_head = sum(p.numel() for p in head_params)
backbone_lr = LR * BACKBONE_LR_RATIO

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": backbone_lr},
    {"params": head_params,     "lr": LR},
], weight_decay=WEIGHT_DECAY)

print(f"  pos_weight={pos_weight.item():.3f}")
print(f"  head_LR={LR}  backbone_LR={backbone_lr}  (ratio = {BACKBONE_LR_RATIO})")
print(f"  backbone params: {n_back/1e6:.1f}M    head params: {n_head/1e6:.2f}M")
print(f"  MIL_W={MIL_WEIGHT if USE_MIL_LOSS else 0.0}  epochs={EPOCHS}  batch={BATCH_SIZE}")
print(f"  n_train_batches={len(train_loader)}  n_val_batches={len(val_loader)}")

criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[backbone_lr, LR],
    steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1,
)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None


# === Diagnostic 8: LR schedule preview (uses dummy params; does NOT touch training scheduler) ===
try:
    _preview_opt = torch.optim.AdamW([
        {"params": [torch.zeros(1, requires_grad=True)], "lr": backbone_lr},
        {"params": [torch.zeros(1, requires_grad=True)], "lr": LR},
    ], weight_decay=WEIGHT_DECAY)
    _preview_sch = torch.optim.lr_scheduler.OneCycleLR(
        _preview_opt, max_lr=[backbone_lr, LR],
        steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1,
    )
    _bb_lrs, _hd_lrs = [], []
    _n_steps = len(train_loader) * EPOCHS
    for _ in range(_n_steps):
        _bb_lrs.append(_preview_opt.param_groups[0]["lr"])
        _hd_lrs.append(_preview_opt.param_groups[1]["lr"])
        _preview_sch.step()

    fig, ax = plt.subplots(figsize=(10, 3))
    _x = np.arange(_n_steps)
    ax.plot(_x, _bb_lrs, label=f"backbone (max={backbone_lr:.1e})", color="tab:blue")
    ax.plot(_x, _hd_lrs, label=f"head (max={LR:.1e})",              color="tab:red")
    for _e in range(EPOCHS + 1):
        ax.axvline(_e * len(train_loader), ls=":", color="gray", alpha=0.3)
    ax.set_yscale("log")
    ax.legend(); ax.grid(True, alpha=0.4)
    ax.set(title=f"OneCycleLR schedule preview (pct_start=0.1, {EPOCHS} epochs × {len(train_loader)} steps)",
           xlabel="optimizer step (dotted = epoch boundary)", ylabel="LR (log)")
    plt.tight_layout()
    plt.savefig("/kaggle/working/lr_schedule.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  Saved lr_schedule.png  (peak backbone={max(_bb_lrs):.2e}, peak head={max(_hd_lrs):.2e})")
    del _preview_opt, _preview_sch, _bb_lrs, _hd_lrs
except Exception as _e:
    print(f"  [warn] LR schedule preview skipped: {_e}")


# === Train + val loop with rich diagnostics ===
ckpt_path = OUT_DIR / "best_val.pt"
best_val_auc_cell = -float("inf")
best_epoch = -1
best_val_data = None  # cell-level arrays captured at best epoch (for histograms + JSON)
history = []
gpu_peak_reported = False
for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_cell, tr_mil, tr_auc = run_epoch_train(
        model, train_loader, optimizer, scaler, criterion_cell, sched,
        pos_weight=pos_weight)
    val = run_epoch_val(model, val_loader)
    val_auc_cell = val["auc_cell"]
    val_auc_pat  = val["auc_pat"]
    dt = time.time() - t0

    is_best = val_auc_cell > best_val_auc_cell
    star = ""
    if is_best:
        best_val_auc_cell = val_auc_cell
        best_epoch = ep
        best_val_data = val  # capture cell-level data for later analysis
        torch.save({
            "model": model.state_dict(),
            "epoch": ep,
            "args": {"dropout": DROPOUT, "backbone": BACKBONE,
                     "val_auc_cell": val_auc_cell, "val_auc_pat": val_auc_pat,
                     "val_patients": present_val,
                     "per_patient": val["per_patient"]},
        }, ckpt_path)
        star = "  ** new best **"

    # Diagnostic 1: per-patient val mean-prob in the same line.
    per_pat_str = "  ".join(
        f"pat_{pid}({'C' if d['label']==1 else 'H'}):{d['mean_prob']:.3f}"
        for pid, d in sorted(val["per_patient"].items())
    )
    print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f}  tr_auc {tr_auc:.4f}  "
          f"val_auc_cell {val_auc_cell:.4f}  val_auc_pat {val_auc_pat:.4f}  "
          f"| {per_pat_str}  | {dt:.1f}s{star}")

    history.append({
        "epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell, "tr_mil": tr_mil,
        "tr_auc": tr_auc, "val_auc_cell": val_auc_cell, "val_auc_pat": val_auc_pat,
        "per_patient": val["per_patient"],
        "time": dt,
    })

    # Diagnostic 3: one-time GPU peak memory after epoch 0.
    if ep == 0 and torch.cuda.is_available() and not gpu_peak_reported:
        peak     = torch.cuda.max_memory_allocated() / (1024**3)
        reserved = torch.cuda.max_memory_reserved()  / (1024**3)
        total    = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"  [mem] peak GPU after ep 0: alloc={peak:.2f}GB  "
              f"reserved={reserved:.2f}GB  / total={total:.1f}GB ({peak/total:.0%})")
        torch.cuda.reset_peak_memory_stats()
        gpu_peak_reported = True

# === Diagnostic 2: best-vs-last + overfitting analysis ===
best = history[best_epoch]
last = history[-1]
print(f"\n=== Training complete ===")
print(f"Best epoch summary:")
print(f"  best  ep {best_epoch:>2d} : tr_auc {best['tr_auc']:.4f}  "
      f"val_auc_cell {best['val_auc_cell']:.4f}  "
      f"gap={best['tr_auc']-best['val_auc_cell']:+.4f}")
print(f"  last  ep {len(history)-1:>2d} : tr_auc {last['tr_auc']:.4f}  "
      f"val_auc_cell {last['val_auc_cell']:.4f}  "
      f"gap={last['tr_auc']-last['val_auc_cell']:+.4f}")
delta = best['val_auc_cell'] - last['val_auc_cell']
print(f"  delta best vs last val_auc_cell = {delta:+.4f}   <- gain from val tracking")

print(f"\nBest-epoch per-patient val:")
for pid, d in sorted(best["per_patient"].items()):
    lab = "CANCER " if d["label"] == 1 else "HEALTHY"
    print(f"  pat_{pid:>2d} ({lab}): mean_prob={d['mean_prob']:.4f}  "
          f"std={d['std_prob']:.4f}  n_cells={d['n_cells']}")

# Overfitting / generalization read.
val_aucs = [e["val_auc_cell"] for e in history]
peak_drop = val_aucs[best_epoch] - val_aucs[-1]
final_gap = last['tr_auc'] - last['val_auc_cell']
if best_epoch < len(history) - 1 and peak_drop > 0.02:
    print(f"\nOverfitting detected:")
    print(f"  val_auc peaks at epoch {best_epoch} ({val_aucs[best_epoch]:.4f})")
    print(f"  val_auc at last epoch {len(history)-1} = {val_aucs[-1]:.4f}  drop = {peak_drop:.4f}")
    print(f"  -> val tracking saved us from a {peak_drop:.3f}-worse submission.")
else:
    print(f"\nNo significant val_auc drop after peak (peak-last = {peak_drop:+.4f}).")
    print(f"  -> val tracking didn't matter much; final-epoch ckpt would have been similar.")
if final_gap > 0.20:
    print(f"  WARNING: final train-val gap = {final_gap:.3f} — heavy overfitting; "
          f"consider more aug / dropout / fewer epochs.")
elif final_gap < 0.02 and last['tr_auc'] < 0.95:
    print(f"  WARNING: final train-val gap = {final_gap:.3f} AND train AUC = {last['tr_auc']:.3f} — "
          f"underfitting; try more capacity or epochs.")

print(f"\nSaved best ckpt:  {ckpt_path}")

with open(OUT_DIR / "history.json", "w") as f:
    json.dump({"backbone": BACKBONE, "val_patients": present_val,
               "best_epoch": best_epoch, "best_val_auc_cell": best_val_auc_cell,
               "history": history}, f, indent=2)

# Diagnostic 6: save per-cell val predictions at best epoch for offline analysis.
val_preds_path = OUT_DIR / "val_predictions.json"
val_records = []
for i in range(len(best_val_data["cell_preds"])):
    val_records.append({
        "name":       str(df_val.iloc[i]["Name"]),
        "patient_id": int(best_val_data["cell_pids"][i]),
        "label":      int(best_val_data["cell_labels"][i]),
        "prediction": float(best_val_data["cell_preds"][i]),
    })
with open(val_preds_path, "w") as f:
    json.dump({"backbone": BACKBONE, "best_epoch": best_epoch,
               "best_val_auc_cell": best_val_auc_cell,
               "val_patients": present_val, "n_cells": len(val_records),
               "cells": val_records}, f, indent=2)
print(f"Saved per-cell val predictions: {val_preds_path}  ({len(val_records)} cells)")

del model, optimizer, sched, scaler, train_loader, train_ds, sampler
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# 2x3 grid of diagnostic plots.
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
epochs = [e["epoch"] for e in history]
val_pids_sorted = sorted(history[0]["per_patient"].keys())

# (1,1) Train losses
ax = axes[0, 0]
ax.plot(epochs, [e["tr_loss"] for e in history], marker="o", color="tab:blue",   label="total")
ax.plot(epochs, [e["tr_cell"] for e in history], marker="s", color="tab:purple", label="cell BCE")
ax.plot(epochs, [e["tr_mil"]  for e in history], marker="^", color="tab:orange", label="MIL")
ax.legend(); ax.grid(True)
ax.set(title="Train losses", xlabel="epoch", ylabel="loss")

# (1,2) Train vs val AUC (cell-level) — primary signal for ckpt selection
ax = axes[0, 1]
ax.plot(epochs, [e["tr_auc"]       for e in history], marker="o", color="tab:green", label="train")
ax.plot(epochs, [e["val_auc_cell"] for e in history], marker="s", color="tab:red",   label="val (cell)")
ax.axvline(best_epoch, ls="--", color="black", alpha=0.5,
           label=f"best ep {best_epoch} ({best_val_auc_cell:.4f})")
ax.axhline(0.5, ls=":", color="gray", alpha=0.5)
ax.legend(); ax.grid(True)
ax.set(title="AUC — cell level (train vs val)", xlabel="epoch", ylabel="AUC")

# (1,3) Train-val gap — overfitting onset tracker  [Diagnostic 5]
ax = axes[0, 2]
gaps = [e["tr_auc"] - e["val_auc_cell"] for e in history]
ax.plot(epochs, gaps, marker="o", color="tab:orange")
ax.axvline(best_epoch, ls="--", color="black", alpha=0.5)
ax.axhline(0.0, ls=":", color="gray", alpha=0.5)
ax.axhline(0.20, ls=":", color="red", alpha=0.4, label="heavy overfit threshold")
ax.legend(); ax.grid(True)
ax.set(title="Train-Val AUC gap (overfitting tracker)",
       xlabel="epoch", ylabel="train_auc - val_auc")

# (2,1) Per-patient val mean-prob trajectories (separation = good)
ax = axes[1, 0]
for pid in val_pids_sorted:
    label = history[0]["per_patient"][pid]["label"]
    trajectory = [h["per_patient"][pid]["mean_prob"] for h in history]
    color = "tab:red" if label == 1 else "tab:blue"
    lab_str = "cancer" if label == 1 else "healthy"
    ax.plot(epochs, trajectory, marker="o", color=color,
            label=f"pat_{pid} ({lab_str})")
ax.axhline(0.5, ls=":", color="gray", alpha=0.5)
ax.axvline(best_epoch, ls="--", color="black", alpha=0.5)
ax.legend(); ax.grid(True)
ax.set(title="Per-patient val mean prob (gap = separation)",
       xlabel="epoch", ylabel="mean P(cancer)")
ax.set_ylim(-0.05, 1.05)

# (2,2) Best-epoch val prediction histograms — per val patient  [Diagnostic 4]
ax = axes[1, 1]
preds      = np.asarray(best_val_data["cell_preds"])
pids_arr   = np.asarray(best_val_data["cell_pids"])
bins       = np.linspace(0, 1, 31)
for pid in val_pids_sorted:
    mask  = pids_arr == pid
    label = best_val_data["per_patient"][pid]["label"]
    color = "tab:red" if label == 1 else "tab:blue"
    lab_str = "cancer" if label == 1 else "healthy"
    ax.hist(preds[mask], bins=bins, alpha=0.55, color=color,
            label=f"pat_{pid} ({lab_str}, n={mask.sum()})")
ax.axvline(0.5, ls=":", color="gray", alpha=0.5)
ax.legend(); ax.grid(True)
ax.set(title=f"Val pred histograms @ best ep {best_epoch}",
       xlabel="P(cancer)", ylabel="cells")

# (2,3) Epoch time
ax = axes[1, 2]
ax.plot(epochs, [e["time"] for e in history], marker="o", color="tab:red")
ax.grid(True)
ax.set(title="Epoch time (s)", xlabel="epoch", ylabel="seconds")

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Compact text summary table (also useful when only the log is available).
print(f"\nEpoch-by-epoch summary:")
hdr = f"  {'ep':>3}  {'tr_auc':>7}  {'val_auc':>7}  {'gap':>7}"
for pid in val_pids_sorted:
    lab = best_val_data["per_patient"][pid]["label"]
    hdr += f"  {('pat_'+str(pid)+'_C' if lab==1 else 'pat_'+str(pid)+'_H'):>10}"
print(hdr)
for e in history:
    gap = e["tr_auc"] - e["val_auc_cell"]
    line = f"  {e['epoch']:>3}  {e['tr_auc']:.4f}  {e['val_auc_cell']:.4f}  {gap:+.4f}"
    for pid in val_pids_sorted:
        line += f"  {e['per_patient'][pid]['mean_prob']:>10.4f}"
    if e["epoch"] == best_epoch:
        line += "  **"
    print(line)

In [ ]:
# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    backbone = args.get("backbone", BACKBONE)
    model = MultimodalClassifier(backbone=backbone, pretrained=False,
                                 dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    print(f"  Loaded ckpt epoch={state.get('epoch')}  backbone={backbone}  "
          f"val_auc_cell={args.get('val_auc_cell', 'n/a')}  "
          f"val_auc_pat={args.get('val_auc_pat', 'n/a')}")
    return model

def predict(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass on test...")
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN}) "
      f"using best-val ckpt at epoch {best_epoch} (val_auc_cell={best_val_auc_cell:.4f})")

t0 = time.time()
preds = predict(ckpt_path, test_loader, tta_scales=scales_to_use)
print(f"  done in {time.time()-t0:.1f}s")

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.4f}, "
      f"min {preds.min():.4f}, max {preds.max():.4f})")
print(f"  <0.05: {(preds < 0.05).mean():.2%}    >0.95: {(preds > 0.95).mean():.2%}")
print(sub.head())
!wc -l /kaggle/working/submission.csv